# APR18 SAOS — frequency / amplitude sweeps and gel-point diagnostics

**Purpose:** Analyse the measured SAOS-type results in the `SS SAOS` sheet of the APR18 workbook. This is a **standalone SAOS notebook**; no raw stress/strain FFT conversion is required.

- **Frequency sweep:** $G'(\omega)$, $G''(\omega)$, $|G^*|$, $|\eta^*|$, $\tan\delta$ and log–log scaling.
- **Amplitude sweep:** $G_1'$, $G_1''$ and third-to-first harmonic stress ratio $I_3/I_1$; high-amplitude points **are not necessarily SAOS**.
- **Winter–Chambon diagnostic:** compare $G'\sim \omega^n$ and $G''\sim\omega^n$ and check whether $\tan\delta$ is frequency independent. **APR18 has no repeated time-resolved frequency sweeps, so its gelation *time* cannot be calculated.**
- **Optional** repeated-frequency dataset: when a separate `time_s, omega_rad_s, G_prime_Pa, G_double_prime_Pa` CSV is provided, analyse its time-resolved Winter–Chambon diagnostics.

### Folder setup
Keep this notebook and the source workbook in the same folder, or put the workbook in `data/` (recommended). Edit `APR18_PATH` in the next cell if it lives elsewhere. All figures are saved as matching **PNG and PDF** files.

## 1. Configuration and shared plot style

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Set explicitly to your workbook path if automatic discovery fails.
APR18_PATH = None  # e.g. Path(r"C:\Users\you\data\APR18 SS SAOS STEPS LAOS UDLAOS.xlsx")
# Optional future time-resolved multi-frequency SAOS CSV; None for APR18-only.
SAOS_TIME_CSV = None  # e.g. Path("data/time_resolved_saos.csv")
OUT_DIR = Path("results") / "saos_apr18"
FIG_DIR = OUT_DIR / "figures"

# Heuristic diagnostic tolerances, NOT validated universal gel-point thresholds.
WC_MIN_FREQUENCIES = 3
WC_MAX_ABS_LOGTAN_SLOPE = 0.10
WC_MAX_LOGTAN_SPREAD = 0.35
WC_MAX_EXPONENT_DIFFERENCE = 0.15

SOURCE_NAME = "APR18 SS SAOS STEPS LAOS UDLAOS.xlsx"
if APR18_PATH is None:
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend((base / "data" / SOURCE_NAME, base / SOURCE_NAME))
    APR18_PATH = next((p for p in candidates if p.is_file()), Path.cwd() / "data" / SOURCE_NAME)
APR18_PATH = Path(APR18_PATH).expanduser().resolve()
if not APR18_PATH.is_file():
    raise FileNotFoundError(f"Cannot find APR18 workbook: {APR18_PATH}. Edit APR18_PATH in this cell.")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Visual design from Conventional_Rheology_Section4_Analysis.ipynb.
plt.rcParams.update({
    "figure.dpi": 115, "savefig.dpi": 250,
    "font.size": 10.5, "axes.labelsize": 11, "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.23,
    "legend.frameon": False,
})
BLUE, ORANGE, GREEN, GREY = '#2166ac', '#d95f02', '#238b45', '#555555'

def save_figure(fig, stem):
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{stem}.{ext}", dpi=250, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {FIG_DIR / (stem + '.png')} (and PDF)")

print("Workbook:", APR18_PATH)
print("Figure directory:", FIG_DIR.resolve())

## 2. Load APR18's **separate** experiments

In `SS SAOS`, workbook columns **E:G** contain the frequency sweep and **I:N** contain an amplitude sweep. These are **not time-series traces**. The amplitude sweep may enter the nonlinear regime; its higher-harmonic terms should not be interpreted as linear SAOS over the entire range.

In [ ]:
def load_apr18_sweeps(workbook):
    raw = pd.read_excel(workbook, sheet_name="SS SAOS", header=None)
    frequency = pd.DataFrame({
        "omega_rad_s": pd.to_numeric(raw.iloc[1:, 4], errors="coerce"),
        "G_prime_Pa": pd.to_numeric(raw.iloc[1:, 5], errors="coerce"),
        "G_double_prime_Pa": pd.to_numeric(raw.iloc[1:, 6], errors="coerce"),
    }).dropna().query("omega_rad_s > 0 and G_prime_Pa > 0 and G_double_prime_Pa > 0")
    frequency = frequency.drop_duplicates("omega_rad_s").sort_values("omega_rad_s").reset_index(drop=True)
    frequency["tan_delta"] = frequency.G_double_prime_Pa / frequency.G_prime_Pa
    frequency["G_star_Pa"] = np.hypot(frequency.G_prime_Pa, frequency.G_double_prime_Pa)
    frequency["complex_viscosity_Pa_s"] = frequency.G_star_Pa / frequency.omega_rad_s
    frequency["loss_angle_deg"] = np.degrees(np.arctan2(frequency.G_double_prime_Pa,
                                                         frequency.G_prime_Pa))

    amplitude = pd.DataFrame({
        "strain_percent": pd.to_numeric(raw.iloc[1:, 8], errors="coerce"),
        "strain_decimal": pd.to_numeric(raw.iloc[1:, 9], errors="coerce"),
        "G1_prime_Pa": pd.to_numeric(raw.iloc[1:, 10], errors="coerce"),
        "G1_double_prime_Pa": pd.to_numeric(raw.iloc[1:, 11], errors="coerce"),
        "G3_prime_Pa": pd.to_numeric(raw.iloc[1:, 12], errors="coerce"),
        "G3_double_prime_Pa": pd.to_numeric(raw.iloc[1:, 13], errors="coerce"),
    }).dropna(subset=["strain_percent", "G1_prime_Pa", "G1_double_prime_Pa"])
    amplitude = amplitude.query("strain_percent > 0").sort_values("strain_percent").reset_index(drop=True)
    i1 = np.hypot(amplitude.G1_prime_Pa, amplitude.G1_double_prime_Pa)
    i3 = np.hypot(amplitude.G3_prime_Pa, amplitude.G3_double_prime_Pa)
    amplitude["I3_over_I1"] = np.where(i1 > 0, i3 / i1, np.nan)
    amplitude["tan_delta_1"] = np.where(amplitude.G1_prime_Pa > 0,
                                          amplitude.G1_double_prime_Pa / amplitude.G1_prime_Pa, np.nan)
    if frequency.empty or amplitude.empty:
        raise ValueError("Missing SAOS frequency or amplitude sweep; inspect sheet SS SAOS.")
    return frequency, amplitude

frequency, amplitude = load_apr18_sweeps(APR18_PATH)
frequency.to_csv(OUT_DIR / "APR18_SAOS_frequency_sweep.csv", index=False)
amplitude.to_csv(OUT_DIR / "APR18_oscillatory_amplitude_sweep.csv", index=False)
print(f"Frequency sweep: {len(frequency)} values, {frequency.omega_rad_s.min():.3g} to {frequency.omega_rad_s.max():.3g} rad/s")
print(f"Amplitude sweep: {len(amplitude)} values, {amplitude.strain_percent.min():.3g} to {amplitude.strain_percent.max():.3g}%")
print("G'' > G' for all measured frequency-sweep points:",
      bool((frequency.G_double_prime_Pa > frequency.G_prime_Pa).all()))
display(frequency.head())

## 3. Frequency sweep — storage and loss moduli

On logarithmic axes, compare the measured storage modulus $G'$ (elastic response) and loss modulus $G''$ (viscous response). A crossover across **frequency** is not a clotting time.

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 5.3))
x = frequency.omega_rad_s
ax.loglog(x, frequency.G_prime_Pa, 'o-', color=BLUE, lw=2, ms=5,
          label=r"Storage modulus $G'$" )
ax.loglog(x, frequency.G_double_prime_Pa, 's-', color=ORANGE, lw=2, ms=5,
          label=r"Loss modulus $G''$" )
ax.fill_between(x, frequency.G_prime_Pa, frequency.G_double_prime_Pa,
                where=frequency.G_double_prime_Pa > frequency.G_prime_Pa,
                color=ORANGE, alpha=0.09)
ax.set(xlabel=r"Angular frequency $\omega$ (rad/s)", ylabel="Modulus (Pa)",
       title="APR18 — measured SAOS frequency sweep")
ax.legend(loc="upper left")
save_figure(fig, "Fig_01_SAOS_moduli_vs_frequency")

## 4. Loss tangent — frequency independence diagnostic

At an ideal Winter–Chambon critical gel, the storage and loss moduli follow a common power law, $G'\sim\omega^n$, $G''\sim\omega^n$, and $\tan\delta=G''/G'$ is independent of frequency. **A single frequency sweep can assess consistency, not gelation time.**

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.6))
ax.semilogx(x, frequency.tan_delta, 'D-', color=GREEN, lw=2, ms=5,
            label=r"$\tan\delta = G''/G'$" )
ax.axhline(1, color=GREY, ls='--', lw=1.2, label=r"$G'=G''$ reference")
ax.set(xlabel=r"Angular frequency $\omega$ (rad/s)",
       ylabel=r"Loss tangent $\tan\delta$", title="APR18 — SAOS loss tangent")
ax.legend(loc="best")
save_figure(fig, "Fig_02_SAOS_loss_tangent_vs_frequency")

## 5. Winter–Chambon **consistency**, not gel-time detection

Fit separate log–log slopes $n'$ and $n''$. The fit describes the **measured range** only; the underlying moduli need not follow a single power law globally. Compare $n'$ and $n''$ and the slope of $\log\tan\delta$ against $\log\omega$. These are exploratory summaries and do not independently validate critical gel behaviour.

In [ ]:
log_w = np.log(frequency.omega_rad_s.to_numpy())
log_gp = np.log(frequency.G_prime_Pa.to_numpy())
log_gpp = np.log(frequency.G_double_prime_Pa.to_numpy())
n_gp, a_gp = np.polyfit(log_w, log_gp, 1)
n_gpp, a_gpp = np.polyfit(log_w, log_gpp, 1)
n_tan = np.polyfit(log_w, np.log(frequency.tan_delta.to_numpy()), 1)[0]
fit_gp = np.exp(a_gp) * x.to_numpy() ** n_gp
fit_gpp = np.exp(a_gpp) * x.to_numpy() ** n_gpp

fig, ax = plt.subplots(figsize=(8.6, 5.3))
ax.loglog(x, frequency.G_prime_Pa, 'o', color=BLUE, label=r"Measured $G'$", ms=5)
ax.loglog(x, frequency.G_double_prime_Pa, 's', color=ORANGE, label=r"Measured $G''$", ms=5)
ax.loglog(x, fit_gp, '--', color=BLUE, lw=1.8, label=fr"$G'$ fit: $n'={n_gp:.2f}$")
ax.loglog(x, fit_gpp, '--', color=ORANGE, lw=1.8, label=fr"$G''$ fit: $n''={n_gpp:.2f}$")
ax.set(xlabel=r"Angular frequency $\omega$ (rad/s)", ylabel="Modulus (Pa)",
       title="APR18 — exploratory power-law scaling")
ax.legend(loc="upper left", fontsize=9)
save_figure(fig, "Fig_03_SAOS_power_law_diagnostics")

wc_summary = pd.DataFrame([{
    "dataset": "APR18 frequency sweep", "num_frequencies": len(frequency),
    "omega_min_rad_s": float(x.min()), "omega_max_rad_s": float(x.max()),
    "exponent_G_prime": n_gp, "exponent_G_double_prime": n_gpp,
    "exponent_difference": n_gp - n_gpp,
    "slope_log_tan_delta": n_tan,
    "tan_delta_min": frequency.tan_delta.min(),
    "tan_delta_max": frequency.tan_delta.max(),
    "tan_delta_log_spread": np.ptp(np.log(frequency.tan_delta.to_numpy())),
    "gel_time_s": np.nan,
    "gel_time_status": "Not identifiable: only one measured frequency sweep (no coagulation time series)"
}])
wc_summary.to_csv(OUT_DIR / "APR18_Winter_Chambon_consistency.csv", index=False)
display(wc_summary.T.rename(columns={0: "APR18"}))

## 6. Frequency-dependent complex modulus and viscosity

Compute $|G^*|=\sqrt{G'^2+G''^2}$ and the magnitude of complex viscosity $|\eta^*|=|G^*|/\omega$. Both are derived from the **frequency sweep** and are not time-dependent measurements.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8.6, 7.1), sharex=True,
                         gridspec_kw={"height_ratios": [1.0, 1.0]})
axes[0].loglog(x, frequency.G_star_Pa, 'o-', color=BLUE, lw=2, ms=5,
               label=r"$|G^*|$")
axes[0].set(ylabel=r"Complex modulus $|G^*|$ (Pa)",
            title="APR18 — derived complex modulus and viscosity")
axes[1].loglog(x, frequency.complex_viscosity_Pa_s, 'D-', color=GREEN,
               lw=2, ms=5, label=r"$|\eta^*|$")
axes[1].set(xlabel=r"Angular frequency $\omega$ (rad/s)",
            ylabel=r"Complex viscosity $|\eta^*|$ (Pa s)")
for ax in axes: ax.legend()
save_figure(fig, "Fig_04_SAOS_complex_modulus_viscosity")

## 7. Amplitude sweep — first-harmonic moduli and nonlinear response

The APR18 amplitude sweep spans very large strains. Plot first-harmonic moduli $G_1'$ and $G_1''$ and the relative third-harmonic signal $I_3/I_1$. Do **not** assume the entire amplitude sweep is linear SAOS. Without a validated linear-viscoelastic strain threshold, this notebook does not declare one.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8.6, 7.1), sharex=True,
                         gridspec_kw={"height_ratios": [1.35, 1.0]})
ax1, ax2 = axes
ax1.semilogx(amplitude.strain_percent, amplitude.G1_prime_Pa, 'o-', color=BLUE,
             lw=1.9, ms=5, label=r"$G'_1$")
ax1.semilogx(amplitude.strain_percent, amplitude.G1_double_prime_Pa, 's-', color=ORANGE,
             lw=1.9, ms=5, label=r"$G''_1$")
ax1.set(ylabel="First-harmonic modulus (Pa)",
        title="APR18 — oscillatory amplitude sweep (not all points are SAOS)")
ax1.legend()
ax2.semilogx(amplitude.strain_percent, amplitude.I3_over_I1, 'D-',
             color=GREEN, lw=1.9, ms=5, label=r"$I_3/I_1$")
ax2.set(xlabel="Applied strain amplitude (%)", ylabel=r"Stress harmonic ratio $I_3/I_1$")
ax2.legend()
save_figure(fig, "Fig_05_SAOS_amplitude_and_harmonics")

## 8. Optional: repeated multi-frequency SAOS over coagulation time

The APR18 workbook **does not contain the required time dimension**. If you later provide a repeated-frequency dataset, set `SAOS_TIME_CSV` in the configuration cell.

CSV format: `time_s`, `omega_rad_s`, `G_prime_Pa`, `G_double_prime_Pa`, optionally `sweep_id`. Each `sweep_id` should represent measurements taken sufficiently close in coagulation time that the material state can be considered approximately unchanged. If `sweep_id` is absent, identical `time_s` values will be grouped. At least **three distinct frequencies** are required per group and at least three time groups are needed to look for a trajectory.

Candidate Winter–Chambon consistency is **not** a validated critical gel time: it requires carefully controlled multi-frequency measurements in the linear regime and evidence of a developing gel. This code reports diagnostic flags rather than claiming an established gel point.

In [ ]:
def read_time_resolved_saos(csv_path):
    d = pd.read_csv(csv_path)
    needed = {"time_s", "omega_rad_s", "G_prime_Pa", "G_double_prime_Pa"}
    if not needed.issubset(d.columns):
        raise ValueError(f"Missing time-resolved SAOS CSV columns: {sorted(needed - set(d.columns))}")
    for c in needed:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=sorted(needed))
    d = d[(d.omega_rad_s > 0) & (d.G_prime_Pa > 0) & (d.G_double_prime_Pa > 0)].copy()
    d["tan_delta"] = d.G_double_prime_Pa / d.G_prime_Pa
    if "sweep_id" not in d.columns:
        d["sweep_id"] = d.time_s  # rows acquired at the same nominal time
    return d


def wc_over_time(saos):
    rows = []
    for sweep_id, group in saos.groupby("sweep_id", sort=False):
        # Multiple measurements at same frequency within a sweep: average them.
        a = (group.groupby("omega_rad_s", as_index=False)
             [["G_prime_Pa", "G_double_prime_Pa", "time_s"]].mean()
             .sort_values("omega_rad_s"))
        if len(a) < WC_MIN_FREQUENCIES:
            continue
        logw = np.log(a.omega_rad_s.to_numpy())
        loggp = np.log(a.G_prime_Pa.to_numpy())
        loggpp = np.log(a.G_double_prime_Pa.to_numpy())
        logtan = loggpp - loggp
        slope_tan = float(np.polyfit(logw, logtan, 1)[0])
        n1 = float(np.polyfit(logw, loggp, 1)[0])
        n2 = float(np.polyfit(logw, loggpp, 1)[0])
        spread = float(np.ptp(logtan))
        rows.append({"sweep_id": sweep_id,
                     "time_mid_s": float(a.time_s.mean()),
                     "time_span_s": float(group.time_s.max() - group.time_s.min()),
                     "n_frequencies": len(a),
                     "slope_log_tan_delta": slope_tan,
                     "log_tan_delta_spread": spread,
                     "exponent_G_prime": n1,
                     "exponent_G_double_prime": n2,
                     "exponent_difference_abs": abs(n1 - n2),
                     "omega_span_ratio": float(a.omega_rad_s.max() / a.omega_rad_s.min()),
                     "candidate_consistency": bool(
                         abs(slope_tan) <= WC_MAX_ABS_LOGTAN_SLOPE and
                         spread <= WC_MAX_LOGTAN_SPREAD and
                         abs(n1-n2) <= WC_MAX_EXPONENT_DIFFERENCE)})
    return pd.DataFrame(rows).sort_values("time_mid_s").reset_index(drop=True) if rows else pd.DataFrame()

if SAOS_TIME_CSV is None:
    print("No separate time-resolved SAOS CSV configured. APR18 gelation time: NOT IDENTIFIABLE.")
else:
    dyn = read_time_resolved_saos(Path(SAOS_TIME_CSV))
    diag = wc_over_time(dyn)
    if diag.empty:
        print("Insufficient independent frequencies per sweep; no WC time diagnostics computed.")
    else:
        diag.to_csv(OUT_DIR / "SAOS_Winter_Chambon_time_diagnostics.csv", index=False)
        display(diag)
        fig, axes = plt.subplots(2, 1, figsize=(9.0, 7.1), sharex=True)
        axes[0].plot(diag.time_mid_s, diag.slope_log_tan_delta, 'o-', color=BLUE, lw=2,
                     label=r"Slope $d\log\tan\delta/d\log\omega$")
        axes[0].axhline(0, color=GREY, ls='--', lw=1.2)
        axes[0].axhspan(-WC_MAX_ABS_LOGTAN_SLOPE, WC_MAX_ABS_LOGTAN_SLOPE,
                        color=GREEN, alpha=.09)
        axes[0].set(ylabel="Log loss-tangent slope",
                    title="Time-resolved SAOS — Winter–Chambon consistency diagnostics")
        axes[0].legend()
        axes[1].plot(diag.time_mid_s, diag.log_tan_delta_spread, 'D-', color=ORANGE,
                     lw=1.8, label=r"Spread of $\log\tan\delta$")
        axes[1].axhline(WC_MAX_LOGTAN_SPREAD, color=GREY, ls='--', lw=1.2,
                        label="Illustrative tolerance")
        mask = diag.candidate_consistency
        axes[1].scatter(diag.loc[mask, "time_mid_s"],
                        diag.loc[mask, "log_tan_delta_spread"],
                        color=GREEN, s=75, zorder=5, label="Candidate-consistent sweep")
        axes[1].set(xlabel="Nominal coagulation time (s)",
                    ylabel=r"Log $\tan\delta$ spread")
        axes[1].legend()
        save_figure(fig, "Fig_06_time_resolved_WC_diagnostics")
        if len(diag) < 3:
            print("Fewer than three time groups: no trajectory or gel-time inference possible.")
        print("Consistency flag count:", int(diag.candidate_consistency.sum()))
        print("Important: flags are NOT confirmed critical gel points.")

## 9. Key interpretation

- **APR18 frequency sweep:** quantifies frequency-dependent elastic/viscous responses. In the measured range, inspect which modulus dominates and how $\tan\delta$ varies with frequency.
- **APR18 amplitude sweep:** probes departure from linear behaviour. Harmonic components at large deformation should not be labelled simple SAOS measurements.
- **Winter–Chambon:** with APR18, report *consistency of one measured frequency sweep* only. No repeated sweeps during coagulation were supplied, so **no APR18 gelation time** can be reported.
- **If supplied repeated SAOS:** use the optional section for diagnostic trajectories, checking that all frequencies within each sweep represent approximately the same material state.

Outputs: `results/saos_apr18/` (CSV summary files) and `results/saos_apr18/figures/` (matching 250-dpi PNG and vector PDF).